# Neural networks with `torch.nn`

**Lab 5 · Notebook 2 of 3**

Complete **`Week_05_nns_in_pytorch.ipynb`** first. Run this notebook from the top in **Google Colab**. It recreates the noisy-XOR dataset from the first notebook: four Gaussian clouds, with diagonally opposite clouds sharing a class label. Each observation receives the class label assigned to its generating cloud. No GPU is needed. Blue circles show class 0 and orange triangles show class 1.

In the previous notebook, we updated every parameter tensor by name. That becomes repetitive and error-prone as a network grows. Here we organise those same calculations into reusable modules and collect their parameter tensors for a common training loop. The network architecture, loss and gradient-descent rule remain the same.

As in the first notebook, `X` contains the original input features, with one observation per row. Unlike the design matrix in Lab 4, it has no additional column of ones, because we add the biases separately.

We also reuse `loss_from_scores(scores)` from the first notebook. Recall that the mathematical log-loss takes a probability and a label, while this helper evaluates the sum

$$
\sum_{i=1}^{N}\ell_{\mathrm{log}}\bigl(\sigma(s_i),y_i\bigr),
$$

using the fixed labels in the tensor `y` defined in the setup cell. The helper uses `logsigmoid` to evaluate the required logarithms directly, without calculating the probabilities as intermediate values. We therefore pass scores to it, without first applying a sigmoid. The first notebook explains the equality between this calculation and the probability-based definition.

We first write small teaching versions of **`Linear`**, **`ReLU`**, and **`Sequential`**. They show the calculations and parameter collection we need here. They are not complete replacements for `torch.nn`.

Recall the callable objects from Lab 2: `__init__` sets up the object's state, while calling the instance runs `__call__`. An affine transformation can therefore store its parameters and be used as `layer(x)`. A whole network can be an object too. We will call these individual software components **modules**. In the notes, one hidden layer combines an affine transformation with ReLU, so two modules together perform its calculation.

Finally, we will use the **`nn.Module`** template to define our own PyTorch layers and networks. It uses the subclassing syntax already shown in Lab 2. The subclassing structure is provided as a template. The inheritance machinery remains non-examinable, as in Lab 2. Focus on the parameters, the calculation in `forward`, and the training loop.


This notebook includes a short question and a small coding exercise. Try each one before opening its explanation or answer. To run code from a supplied answer, copy it into the exercise cell and run that cell. Opening the answer box does not execute the code. If you get stuck, use the supplied answer and continue. The later examples do not depend on your exercise solution.


In [ ]:
import torch as t
import matplotlib.pyplot as plt
t.manual_seed(1)  # Fix the random seed for this notebook.

N = 100
# Same noisy-XOR observations as the first notebook: 25 per Gaussian group.
# spread is the standard deviation in each input coordinate.
spread = 0.45
# Diagonally opposite groups share a class. Each label is the class assigned to the generating cloud.
x_class_0 = t.cat([
    t.randn(N // 4, 2) * spread + t.tensor([-1.0, -1.0]),
    t.randn(N // 4, 2) * spread + t.tensor([ 1.0,  1.0])
], dim=0)
x_class_1 = t.cat([
    t.randn(N // 4, 2) * spread + t.tensor([-1.0,  1.0]),
    t.randn(N // 4, 2) * spread + t.tensor([ 1.0, -1.0])
], dim=0)
X = t.cat([x_class_0, x_class_1], 0)              # Shape: (100, 2).
y = t.cat([t.zeros(N // 2, 1), t.ones(N // 2, 1)], 0)  # Shape: (100, 1).

# Show observed classes using the same colours and markers as in Lab 4.
plt.figure()
plt.scatter(x=x_class_0[:, 0], y=x_class_0[:, 1], color='tab:blue', label='Class 0')
plt.scatter(x=x_class_1[:, 0], y=x_class_1[:, 1], color='tab:orange', marker='^', label='Class 1')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.legend()

def loss_from_scores(scores):
    # scores contains the network's logits, not probabilities or weights.
    # Evaluate sum_i ell_log(sigmoid(score_i), y_i), using the fixed labels y.
    # logsigmoid computes the logarithms directly, without forming probabilities.
    return -(y * t.nn.functional.logsigmoid(scores)
             + (1 - y) * t.nn.functional.logsigmoid(-scores)).sum()

## 1. An affine module

`MyLinear(in_features, out_features)` creates a module that computes `x @ W + b`. With the bias included, this is an **affine** transformation. PyTorch calls it a *linear layer*.

`__init__` creates its weights and biases once and stores them as `self.W` and `self.b`, using the letters from the notes. Each module has its own pair of tensors. `__call__` then applies the transformation using their current values. The additional method **`parameters()`** returns those two tensors so that our training loop can update them.

As in the first notebook, `b` is stored as a one-dimensional tensor of shape `(out_features,)`, although the notes write the bias as a row. When `x` contains several input rows, broadcasting adds the same bias entries to each output row.


In [ ]:
class MyLinear:
    def __init__(self, in_features, out_features):
        """
        Initialise this layer's weights and biases once, when it is created.
        For example: lin = MyLinear(3, 4).
        """
        self.W = t.randn(in_features, out_features, requires_grad=True)
        self.b = t.randn(out_features, requires_grad=True)

    def __call__(self, x):
        """
        Apply the layer to x, for example:
            input = t.randn(5, 3)
            lin = MyLinear(3, 4)
            output = lin(input)
        """
        # (batch, in_features) @ (in_features, out_features), then add the bias.
        return x @ self.W + self.b

    def parameters(self):
        """Return this layer's parameter tensors for the gradient-descent loop."""
        return [self.W, self.b]

In [ ]:
# Create a layer: three features per input, four features per output.
in_features = 3
out_features = 4
batch = 5  # Number of input rows in this example.
lin = MyLinear(in_features, out_features)

input = t.randn(batch, in_features)  # Shape: (5, 3).
output = lin(input)                 # Calls lin.__call__; shape: (5, 4).
print('Output shape:')
print(output.shape)

# These are the tensors stored on lin, not newly initialised parameters.
print('Parameters:')
print(lin.parameters())

## 2. A ReLU module

`MyReLU` applies `t.relu` to its input. For an input row, this implements the vector transformation $\operatorname{\mathbf{ReLU}}$ introduced in the notes. For a batch of rows, the operation applies ReLU to every entry and preserves the tensor's shape.

There are no weights or biases to initialise or learn, so we do not need a custom `__init__`, and `parameters()` returns an empty list.

ReLU still participates in the gradient calculation, although it has no parameters of its own. Together with the preceding affine module, it will form a hidden layer in the terminology of our notes.


In [ ]:
class MyReLU:
    def __call__(self, x):
        return t.relu(x)  # Replace negative entries by zero.

    def parameters(self):
        return []  # No parameter tensors to update for this module.

In [ ]:
in_features = 3
batch = 5
relu = MyReLU()  # Create the module object.
input = t.randn(batch, in_features)
print(relu(input))         # Apply the module. The shape remains (5, 3).
print(relu.parameters())  # An empty list: [].

## 3. Put the modules in sequence

`MySequential` stores a list of module objects. When called, it passes the input through them in order, so the output of one module becomes the input to the next.

Its `parameters()` method collects the parameter tensors from all the modules. These are the tensors stored on those modules, so updating them updates the network. A `MyReLU` contributes no parameters because its list is empty.


In [ ]:
class MySequential:
    def __init__(self, modules):
        """Store a list of modules, such as linear layers and ReLUs."""
        self.modules = modules

    def __call__(self, x):
        """Apply each module in order. Two modules give modules[1](modules[0](x))."""
        for mod in self.modules:
            x = mod(x)  # Feed this module's output to the next module.
        return x

    def parameters(self):
        """Collect the parameter tensors from every module."""
        full_list = []
        for mod in self.modules:
            for param in mod.parameters():
                full_list.append(param)  # Add this tensor to the list.
        return full_list

Now build the same **2 → 100 → 1** network as in the first notebook. The list passed to `MySequential` contains three modules: an affine transformation, ReLU, and another affine transformation. The first two together form hidden layer 1, while the last forms output layer 2. Therefore, these three software modules implement the two layers in our mathematical description.

The first affine module stores the parameters denoted by $\boldsymbol{W}_1$ and $\boldsymbol{b}_1$ in the notes, and the second stores $\boldsymbol{W}_2$ and $\boldsymbol{b}_2$. Both use the attribute names `W` and `b`, because each module stores its own tensors.

The final line below uses a **list comprehension**: it reads `p.shape` for each parameter tensor `p` and collects those shapes into a list.


In [ ]:
input_features = 2
hidden_features = 100
output_features = 1
net = MySequential([
    MyLinear(input_features, hidden_features),
    MyReLU(),
    MyLinear(hidden_features, output_features)
])
print([p.shape for p in net.parameters()])

### Pause and connect: which parameters change the hidden features?

The network above performs the same calculations as the explicit network in the first notebook. The following lines recall that earlier implementation. Read them to trace the dependencies when answering the question:

```python
a1 = X @ W1 + b1
phi1 = t.relu(a1)
scores = phi1 @ W2 + b2
```

Here `W1` and `b1` refer to the tensors stored on the first affine module, while `W2` and `b2` refer to those stored on the second.

Keep the input observations `X` fixed. Which of these parameter tensors can change the hidden activations `phi1`? If we change only `W2` and `b2`, which of `a1`, `phi1`, and `scores` can change? Explain your answer by following the three calculations.


<details>
<summary>Show explanation — which parameters change the hidden features?</summary>

The first calculation uses `W1` and `b1` to form `a1`, and ReLU then turns `a1` into `phi1`. Therefore, changing these first-layer parameters can change the hidden activations for the same input observations. It can also change the feature functions that the hidden layer computes.

However, `W2` and `b2` appear only in the final calculation. If we change only these output-layer parameters, `a1` and `phi1` stay unchanged, while `scores` can change. In that case, we are changing how fixed hidden features contribute to the output. This is the distinction between adjusting the features themselves and adjusting their final combination.

</details>


Train with the same loss, learning rate, and 1,000 updates as before. The difference is that **one loop over `net.parameters()`** now updates all four parameter tensors.

`no_grad()` has the same role as in the first notebook. The parameters are retained between iterations, while their stored gradients are cleared after each update. The losses are recomputed and printed **after** updates 100, 200, ..., 1,000, so the last printed value describes the trained network.

In [ ]:
learning_rate = 0.0001
for step in range(1, 1001):
    L = loss_from_scores(net(X))
    L.backward()

    with t.no_grad():
        for param in net.parameters():
            param.add_(-learning_rate * param.grad)
            param.grad.zero_()  # Clear the gradient, not the learned parameter.

    if step % 100 == 0:
        with t.no_grad():
            print("After", step, "updates:", loss_from_scores(net(X)).item())


In [ ]:
def plot_decision_boundary(net, X, y):
    # Non-examinable: display the fitted classifier and the observed class labels.
    # Use the same data-dependent limits for every network, including all observations.
    x1_min = X[:, 0].min().item() - 0.5
    x1_max = X[:, 0].max().item() + 0.5
    x2_min = X[:, 1].min().item() - 0.5
    x2_max = X[:, 1].max().item() + 0.5
    x1_grid, x2_grid = t.meshgrid(t.linspace(x1_min, x1_max, 1000),
                                t.linspace(x2_min, x2_max, 1000),
                                indexing="ij")
    X_grid = t.stack([x1_grid, x2_grid], dim=-1)

    # We only need predictions for this plot, not their gradients.
    with t.no_grad():
        c_grid = (net(X_grid) > 0)[..., 0]

    plt.figure()
    plt.contour(x1_grid, x2_grid, c_grid, levels=[0.5], colors='black', linewidths=0.7)
    class_0 = y[:, 0] == 0
    class_1 = y[:, 0] == 1
    plt.scatter(X[class_0, 0], X[class_0, 1], color='tab:blue', label='Class 0')
    plt.scatter(X[class_1, 0], X[class_1, 1], color='tab:orange', marker='^', label='Class 1')
    plt.xlim(x1_min, x1_max)
    plt.ylim(x2_min, x2_max)
    plt.xlabel('$x_1$')
    plt.ylabel('$x_2$')
    plt.legend()

plot_decision_boundary(net, X, y)

## 4. Use the built-in `torch.nn` modules

PyTorch provides **`nn.Linear`**, **`nn.ReLU`**, and **`nn.Sequential`**. Their objects are callable, and we can iterate over their parameter tensors using `.parameters()`, just as in our training loop above.

The new name `torch_nn_net` keeps this network separate from our handwritten `net`. The next cell prints the parameter shapes of this **new** network.

In [ ]:
import torch.nn as nn  # Use nn as the short name for torch.nn.

torch_nn_net = nn.Sequential(
    nn.Linear(input_features, hidden_features),
    nn.ReLU(),
    nn.Linear(hidden_features, output_features)
)
print([p.shape for p in torch_nn_net.parameters()])

Three differences matter when reading these examples:

* **Weight layout:** our `MyLinear` stores `W` with shape `(in_features, out_features)` and computes `x @ W + b`. The built-in `nn.Linear` instead uses the attribute names `weight` and `bias`. It stores `weight` with shape `(out_features, in_features)` and computes the equivalent `x @ weight.mT + bias`. This explains the different weight shapes just printed, while the input and output shapes remain unchanged.
* **Initialisation:** `nn.Linear` scales its random initialisation using the number of input features, rather than using our unscaled `t.randn` values.
* **Constructor syntax:** `nn.Sequential(module1, module2, module3)` takes the modules as separate arguments. Our `MySequential([module1, module2, module3])` takes one argument: a list.

The network still returns **scores**, so we use the same `loss_from_scores` without adding a final sigmoid. Different random initial parameters mean that these training losses need not match the preceding network's losses.


In [ ]:
learning_rate = 0.0001
for step in range(1, 1001):
    L = loss_from_scores(torch_nn_net(X))
    L.backward()

    with t.no_grad():
        for param in torch_nn_net.parameters():
            param.add_(-learning_rate * param.grad)
            param.grad.zero_()

    if step % 100 == 0:
        with t.no_grad():
            print("After", step, "updates:", loss_from_scores(torch_nn_net(X)).item())

plot_decision_boundary(torch_nn_net, X, y)


### Your turn: construct a smaller network

Use `nn.Sequential`, `nn.Linear`, and `nn.ReLU` to build a fresh network named `small_net` with **two input features, three hidden units, and one output score**. The hidden layer should apply an affine transformation followed by ReLU, and the output layer should form an affine combination of the three hidden activations.

Before writing the code, predict the shape of the scores when this network receives the existing input matrix `X`. Also predict the shapes of its four parameter tensors, remembering how `nn.Linear` stores its weight matrix.

Then construct `small_net`, compute `small_scores = small_net(X)`, and print `small_scores.shape`. Print the parameter shapes using the same list-comprehension pattern as in the preceding example, with `small_net.parameters()`. Compare the printed shapes with your predictions.

Use the new names `small_net` and `small_scores` throughout this exercise, so the networks from the worked examples retain their own names and parameters. This exercise is complete after one forward pass and the shape checks. We will continue with the next worked example afterwards.


In [ ]:
# Write your predicted output and parameter shapes here as comments.

# Construct small_net using the three module types from the preceding example.

# Compute small_scores for X, then print its shape and the parameter shapes.


<details>
<summary>Show answer — construct a smaller network</summary>

```python
small_net = nn.Sequential(
    nn.Linear(2, 3),
    nn.ReLU(),
    nn.Linear(3, 1)
)
small_scores = small_net(X)
print(small_scores.shape)
print([p.shape for p in small_net.parameters()])
```

For the dataset in this notebook, the output is:

```text
torch.Size([100, 1])
[torch.Size([3, 2]), torch.Size([3]), torch.Size([1, 3]), torch.Size([1])]
```

Each of the `N = 100` input rows produces one output score, so `small_scores` has shape `(N, 1)`. The first affine module has three outputs and two inputs. Since `nn.Linear` stores its weight matrix with one row per output, its weights have shape `(3, 2)` and its bias has shape `(3,)`. The second affine module combines three hidden activations into one score, so its weights have shape `(1, 3)` and its bias has shape `(1,)`. ReLU adds no parameter tensors and leaves the number of hidden values unchanged.

</details>


## 5. Define your own affine `torch.nn` module

We will rewrite the familiar affine calculation once more to see how `nn.Module` automatically collects its parameters, replacing the `parameters()` method we previously wrote ourselves.

We can also write an affine module as a subclass of **`nn.Module`**. As before, `__init__` sets up the instance's state. Three details make this version work with PyTorch:

* **`super().__init__()`** runs the `nn.Module` initialiser on this instance before we add its parameters.
* **`nn.Parameter(...)`** makes a tensor a trainable parameter (`requires_grad=True` by default). Assigning it as an attribute of this module also registers it, so the inherited `.parameters()` method finds it automatically.
* We write the computation in **`forward`**, not `__call__`. `nn.Module` already supplies the calling behaviour: calling `layer(x)` runs our `forward` through that mechanism. Continue to call the object as `layer(x)`.

Thus we no longer need to write our own `parameters()` method. This module keeps our original weight layout and the attribute names `W` and `b`.

In [ ]:
class My_torchnn_Linear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()  # Set up this instance as an nn.Module first.

        # Register trainable parameter tensors on this module.
        self.W = nn.Parameter(t.randn(in_features, out_features))
        self.b = nn.Parameter(t.randn(out_features))

    def forward(self, x):
        return x @ self.W + self.b

## 6. Define a whole network as an `nn.Module`

An `nn.Module` can contain other modules. Here, `self.layer1` and `self.layer2` store the affine transformations for layers 1 and 2. PyTorch registers them as **submodules**, so `mynet.parameters()` will include their parameters too. In particular, `self.layer1.W` and `self.layer1.b` store the first layer's parameters, and `self.layer2.W` and `self.layer2.b` store the second layer's parameters.

As usual, the constructor arguments determine the layer sizes. In `forward`, the names `a1`, `phi1`, and `scores` follow the three calculations in the notes: first the preactivations, then the hidden activations, and finally the output scores. When the input is `X`, each row of `phi1` contains $\boldsymbol{\phi}_1(\boldsymbol{x}_i)$ for one observation. For our **2 → 100 → 1** network, `a1` and `phi1` therefore have shape `(N, 100)`, while `scores` has shape `(N, 1)`.

The hidden layer consists of `self.layer1` followed by ReLU. The output layer is `self.layer2`. Calling the network applies these stored transformations, so it does not recreate the modules or reset their parameters.

Here `t.relu(a1)` performs the same parameter-free calculation as an `nn.ReLU()` module.


In [ ]:
class MyNet(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super().__init__()

        # Use this constructor's arguments, not similarly named global variables.
        self.layer1 = My_torchnn_Linear(in_features, hidden_features)
        self.layer2 = My_torchnn_Linear(hidden_features, out_features)

    def forward(self, x):
        a1 = self.layer1(x)
        phi1 = t.relu(a1)
        scores = self.layer2(phi1)
        return scores

For a simple chain of layers, `nn.Sequential` is shorter. Writing `forward` gives us direct control over the steps. For example, we could temporarily print the shape of `phi1` there.

Train the network as before. **`mynet.zero_grad()`** clears the stored gradients for all its parameters at once (by default, setting their `.grad` attributes back to `None`). It does **not** reset the learned weights or biases. The next call to `backward()` computes and stores fresh gradients.

After this example, move on to **`Week_05_nn_exercises.ipynb`** to practise these same constructions.

In [ ]:
learning_rate = 0.0001
mynet = MyNet(2, 100, 1)  # Create a fresh network before training.
for step in range(1, 1001):
    L = loss_from_scores(mynet(X))
    L.backward()

    with t.no_grad():
        for param in mynet.parameters():
            param.add_(-learning_rate * param.grad)

    mynet.zero_grad()  # Clear all stored parameter gradients in one call.

    if step % 100 == 0:
        with t.no_grad():
            print("After", step, "updates:", loss_from_scores(mynet(X)).item())

plot_decision_boundary(mynet, X, y)
